In [1]:
#read needed packages, should execute everytime
#version 0.2 (newer than proto, which includes some pre-processing steps)
import pandas as pd
import time
import csv
import re
import ast

In [2]:
#read glycotope settings if there is one
#in alpha version it's hand-written in below cell
try:
    theoglycotopedict = pd.read_csv("glycotope_deducer.csv")
except:
    theoglycotopedict = {}

In [9]:
#hand writing rules of glycotopes
glycotopeheader = ["Name of glycotope feature", "used units", "enzyme list and presence 0,1 or ?", "ion list", "groups"]
#explanations: name: name, used units: no. of units used for calculation, if we can deduce from structure, not from the
#monosaccharide units, then this part will become depreciated. enzyme list: 0 for absence, 1 for presence, "?" for unknown
#ion list = list of ions supporting this epitope, 
#groups: stem = N-glycan core, only used when other starting cores are not found, core: N-glycan cores (1 exclusive for each search)
#ext: extension possible, ter: terminal epitope, no longer extensible.
#type1: based on type1 lacnac, type2: based on type2 lacnac


#this is defining epitope part
#Maybe it's better to keep values in dict as the same datatype
NG_core= ["NG core", {"H":3, "N":2}, {"MGAT1":1 ,"MGAT2":1, "MAN2A1":1 ,"MAN2A2":1, "unknown enzyme": '?'}, [1234.56, 888.88], "stem"]
Bisecting_NG_core= ["Bisecting NG core", {"H":3, "N":3}, {"MGAT3": 1}, [1234.56, 888.88], "core"]
Triante_NG_core= ["Tri-antennary N-glycan core", {"H":6, "N":5}, {"MGAT4":1 ,"MGAT5":1}, [1234.56, 888.88], "core"]
NG_elongationGal = ["First and elongation of Galactose at NG ", {"H":1}, {"B4GALT1":1, "B4TALT2~7": 1}, [187.0965, 209.1227], "ext"] #re?
corefuc = ["N-glycan core Fucosylation", {"F":1}, {"FUT8":1}, [452.2490, 697.3753], "ter"]
LacNAc1 = ["Type 1 LacNAc", {"H":1, "N":1}, {"B3GALT2":1, "B3GALT5":1}, [464], "ext, type1"]
LacNAc2 = ["Type 2 LacNAc", {"H":1, "N":1}, {"B4GALT1~7":1}, [464, 432], "ext, type2"]
H_type1 = ["a2Fuc-Type1 LNAc", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]
H_type2 = ["a2Fuc-Type2 LNAc", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type2"]
LeA = ["Lewis A antigen", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ext, ter, type1"]#b3-Gal type1 w a4-Fuc
LeX = ["Lewis X antigen", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ext, ter, type2"]#b4-Gal type2 w a3-Fuc
LeB = ["Lewis B antigen", {"F":2, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]#b3-Gal type1 w a4-Fuc, a2-Fuc at Gal
LeY = ["Lewis Y antigen", {"F":2, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type2"]#b4-Gal type2 w a3-Fuc, a2-Fuc at Gal
Neu5Ac = ["Neu5Ac sialic acid", {"S": 1}, {"CMAS": 1}, [376],  "ext, ter"]
Neu5Gc = ["Neu5Gc sialic acid", {"G": 1}, {"CMAH": 1}, [406], "ext, ter"]
KDN = ["KDN sialic acid", {"KDN": 1}, {"CMP-KDN synthetase": 1}, [335], "ter"]
Sia3_LNAc1 = ["a2-3-Sialylated Type 1 LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ext, type1"]
Sia6_LNAc1 = ["a2-6-Sialylated Type 1 LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ext, type1"]
Sia3_LNAc2 = ["a2-3-Sialylated Type 2 LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ext, type2"]
Sia6_LNAc2 = ["a2-6-Sialylated Type 2 LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ext, type2"]
zf_GalExtension = ["a1-4 Galx2 + GlcNAc", {"H":2, "N":1}, {"unknown": "?"}, [668], "ext"]
zf_specific_epitope_5Ac = ["a1-4 Galx2 + GlcNAc + a2-3Sia + a1-3Fuc", {"F":1, "H":2, "N":1, "S":1}, {"unknown": "?"}, [1203], "ext"]
zf_specific_epitope_5Gc = ["a1-4 Galx2 + GlcNAc + a2-3Sia-Gc + a1-3Fuc", {"F":1, "H":2, "N":1, "G":1}, {"unknown": "?"}, [1233], "ext"]
LacDiNAc = ["LacDiNAc", {"N":2}, {}, [505], "ext, ter"]
Sia6_LacDiNAc = ["a2-6Sia_LacDiNAc", {"N":2}, {}, [505], "ext, ter"]



def calcmassinepitope():
    print("developing")
#thinking if we can split epitopes into single component
#thinking if we can calculate fragments back from predicted structure (does that make sense in this pipeline?)
#should the ion be unique in all groups to avoid counted multiple times?


glycoenzymelist = [
    glycotopeheader,
    NG_core,
    Bisecting_NG_core,
    NG_elongationGal, 
    corefuc,
    LacNAc1,
    LacNAc2]

tmp = []
for glycoepitopes in glycoenzymelist:
    tmp.append(glycoepitopes)
print(f"tmp is+ {tmp}")
c = pd.DataFrame(tmp)
c.columns = c.iloc[0]
c = c[1:]
#set first row as index column, copied from https://www.statology.org/pandas-set-first-row-as-header/
c


tmp is+ [['Name of glycotope feature', 'used units', 'enzyme list and presence 0,1 or ?', 'ion list', 'groups'], ['NG core', {'H': 3, 'N': 2}, {'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2': 1, 'unknown enzyme': '?'}, [1234.56, 888.88], 'stem'], ['Bisecting NG core', {'H': 3, 'N': 3}, {'MGAT3': 1}, [1234.56, 888.88], 'core'], ['First and elongation of Galactose at NG ', {'H': 1}, {'B4GALT1': 1, 'B4TALT2~7': 1}, [187.0965, 209.1227], 'ext'], ['N-glycan core Fucosylation', {'F': 1}, {'FUT8': 1}, [452.249, 697.3753], 'ter'], ['Type 1 LacNAc', {'H': 1, 'N': 1}, {'B3GALT2': 1, 'B3GALT5': 1}, [464], 'ext, type1'], ['Type 2 LacNAc', {'H': 1, 'N': 1}, {'B4GALT1~7': 1}, [464, 432], 'ext, type2']]


,Name of glycotope feature,used units,"enzyme list and presence 0,1 or ?",ion list,groups
1,NG core,"{'H': 3, 'N': 2}","{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2'...","[1234.56, 888.88]",stem
2,Bisecting NG core,"{'H': 3, 'N': 3}",{'MGAT3': 1},"[1234.56, 888.88]",core
3,First and elongation of Galactose at NG,{'H': 1},"{'B4GALT1': 1, 'B4TALT2~7': 1}","[187.0965, 209.1227]",ext
4,N-glycan core Fucosylation,{'F': 1},{'FUT8': 1},"[452.249, 697.3753]",ter
5,Type 1 LacNAc,"{'H': 1, 'N': 1}","{'B3GALT2': 1, 'B3GALT5': 1}",[464],"ext, type1"
6,Type 2 LacNAc,"{'H': 1, 'N': 1}",{'B4GALT1~7': 1},"[464, 432]","ext, type2"


In [12]:
#this is defining species part (only runs once for each analysis)

zebrafish_glycoT_brain = {"MGAT1":1 ,"MGAT2":1, "MAN2A1":1 ,"MAN2A2":1, "absentenzyme": 0, "unknown": "?"}

#to get possible epitope (only runs once for each analysis)
def findepitopefrom_glycoT(species):
    expression_list = []
    for key, value in species.items():
        if value == 1:
            expression_list.append(key)
    return expression_list
#those not expressed glycoT and unknown glycoT isn't returned in this version, consider if we need a way to recycle and use them

zfdemo = findepitopefrom_glycoT(zebrafish_glycoT_brain)
print(zfdemo)

['MGAT1', 'MGAT2', 'MAN2A1', 'MAN2A2']


In [13]:
c["enzyme list and presence 0,1 or ?"][1]
#print(c.loc[c["enzyme list and presence 0,1 or ?"]).isin(zfdemo)])

{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2': 1, 'unknown enzyme': '?'}

In [18]:
#to get current structure list

#useless since calculation only runs in dictionary...

structure_dict = {"F":1, "H":5 , "N": 4, "S":1}
def getdeducestructure(structure_dict)
    if structure_dict is None:
        struc_tmp = {}
    try:
        Fuc = struc_tmp["F"]
    except KeyError: #keyerror -> Nameerror -> Syntaxerror (traceback)
        Fuc = 0
    try:
        Hex = struc_tmp["H"]
    except KeyError:
        Hex = 0
    try:
        HexNAc = struc_tmp["N"]
    except KeyError:
        HexNAc = 0
    try:
        Neu5Ac = struc_tmp["S"]
    except KeyError:
        Neu5Ac = 0
    try:
        Neu5Gc = struc_tmp["G"]
    except KeyError:
        Neu5Gc = 0
    try:
        KDN = struc_tmp["KDN"]
    except KeyError:
        KDN = 0
    try:
        deduceunitlist = [Fuc, Hex, HexNAc, Neu5Ac, Neu5Gc, KDN]
        #calc = struc_tmp - NG_core[1]
        #need to calculate key value inside the dict, direct add/sub is not valid
    except:
        print("error")
        deduceunitlist = None
    return deduceunitlist

def deducestructure(composition, dealt_epitope):
    if composition is None:
        struc_tmp = [0, 0, 0, 0, 0, 0]
    else:
        struc_tmp = structure_dict
    try:
        #pseudocode:
        #for 
        #calc = struc_tmp - NG_core[1]
        #need to calculate key value inside the dict, direct add/sub is not valid
    except:
        print("error")
        calc = None
    return calc

#deducestructure(structure_dict)

[1, 5, 4, 1, 0, 0]

In [14]:
print(f"This is testing cell")
print(len(c.index))
for i in range(1,len(c.index)+1):
    print(c["enzyme list and presence 0,1 or ?"][i])

This is testing cell
6
{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2': 1, 'unknown enzyme': '?'}
{'MGAT3': 1}
{'B4GALT1': 1, 'B4TALT2~7': 1}
{'FUT8': 1}
{'B3GALT2': 1, 'B3GALT5': 1}
{'B4GALT1~7': 1}


In [39]:
c.loc[1]

0
Name of glycotope feature                                                      NG core
used units                                                            {'H': 3, 'N': 2}
enzyme list and presence 0,1 or ?    {'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2'...
ion list                                                             [1234.56, 888.88]
groups                                                                            stem
Name: 1, dtype: object

In [15]:
#c.loc[c["enzyme list and presence 0,1 or ?"].isin(zfdemo)] is not applicable sice we're wrapping dict+tuples inside the column 
newdfl = []
zzz = 1
for i in c["enzyme list and presence 0,1 or ?"]:
    e =[]
    for key, value in i.items():
        if value == 1:
            e.append(key)
    s = list(set(e) & set(zfdemo))
    if s != []:
        newdfl.append(c.loc[zzz])
        #should be appending the whole row
    else:
        print("There is no enzyme for this epitope")
    zzz +=1
    
newdf = pd.DataFrame(newdfl)

newdf

There is no enzyme for this epitope
There is no enzyme for this epitope
There is no enzyme for this epitope
There is no enzyme for this epitope
There is no enzyme for this epitope


,Name of glycotope feature,used units,"enzyme list and presence 0,1 or ?",ion list,groups
1,NG core,"{'H': 3, 'N': 2}","{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2'...","[1234.56, 888.88]",stem


In [16]:
#write upper one into function
def findpossibleepitope(epitopes, species): 
    foundepitope = []
    index = 1  #need to adjust to correct indexed dataframe
    epitopelist = epitopes
    for i in epitopelist["enzyme list and presence 0,1 or ?"]:
        elist =[] #re-create to iterate through all rows and keep intact structure
        for key, value in i.items():
            if value == 1:
                elist.append(key)
        s = list(set(elist) & set(species))
        if s != []:
            foundepitope.append(epitopelist.loc[index])
            #should be appending the whole row
        else:
            print(f"There is no enzyme for this epitope {epitopelist.loc[index][0]}")
        index +=1

    foundepitopes = pd.DataFrame(foundepitope)
    return foundepitopes

findpossibleepitope(c, zfdemo)

There is no enzyme for this epitope Bisecting NG core
There is no enzyme for this epitope First and elongation of Galactose at NG 
There is no enzyme for this epitope N-glycan core Fucosylation
There is no enzyme for this epitope Type 1 LacNAc
There is no enzyme for this epitope Type 2 LacNAc


,Name of glycotope feature,used units,"enzyme list and presence 0,1 or ?",ion list,groups
1,NG core,"{'H': 3, 'N': 2}","{'MGAT1': 1, 'MGAT2': 1, 'MAN2A1': 1, 'MAN2A2'...","[1234.56, 888.88]",stem


In [26]:
print("this cells includes demo epitopes that don't check enzyme and peaklist")
#assigning core structure priority: largest core -> smallest core -> stem
#if core assignment successful -> try to extend based on possible "ext" epitopes
#if there's sugar unit left when no ext epitopes is possible, try to add "terminal" epitopes and ends of all values at 0
#if the enumeration cannot reach zero, then it means this ISNOT a valid structure
#DO NOT break the enumeration loop since we may get multiple assignment possible

#this step is HIGHLY POTENTIAL to get optimized by any of AI method since they do this kind of task better than human

#core structure
Bisecting_NG_core= ["Bisecting NG core", {"H":3, "N":3}, {"Bisecting enzyme": 1}, [], "core"]
Biante_NG_core= ["Bi-antennary N-glycan core", {"H":5, "N":4}, {}, [], "core"]
Triante_NG_core= ["Tri-antennary N-glycan core", {"H":6, "N":5}, {}, [], "core"]
Tetraante_NG_core = ["Tetra-antennary N-glycan core", {"H":7, "N":6}, {}, [], "core"]
Hybrid_NG_core = ["Bi-antennary hybrid N-glycan core", {"H":6, "N":3}, {}, [], "core"]

#stem structure, not included in the first demo
NG_core= ["NG core", {"H":3, "N":2}, {}, [], "stem"]
NG_elongationGal = ["First and elongation of Galactose at NG ", {"H":1}, {}, [], "ext"] #re?

#higher priority to add core fucosylation if the enzyme is there
corefuc = ["N-glycan core Fucosylation", {"F":1}, {"FUT8":1}, [452.2490, 697.3753], "ter, coreter"]


#calculate larger terminal structure to use out terminal units, from largest one
#maybe we can apply a sort method to do this based on sum of all key values in sugar unit dict

#calculate start from "each arm?"
#another idea here: try to put as much big units as possible for all arms
#for example, 3 arms, S x 3, F x 3, H x 5, N x 5 is left after core substraction

#example2: 2 arms, S x 2, F x 3, H x 2, N x 2

#calculate average S and F in each arm first
# ex1: S = 1, F = 1 while H, N = 1.66
# ex2: S = 1, F = 1.5 while H, N = 1
# when one value is larger than 1, try to assign a larger one epitope first.

#here we consider all same "composition" as one structure to validate our guess.
#in next exp we'll add enzyme list and peak list to compare and make sure which defines it's real possible structure... 


#if ratio N>H, add LDNA if enzyme permits it to do so
LacDiNAc = ["LacDiNAc", {"N":2}, {}, [505], "ext, ter"]

#for zebrafish case, we have certain structure need to be consider first
zf_specific_epitope_5Ac = ["a1-4 Galx2 + GlcNAc + a2-3Sia + a1-3Fuc", {"F":1, "H":2, "N":1, "S":1}, {"unknown": "?"}, [1203], "ext"]
zf_specific_epitope_5Gc = ["a1-4 Galx2 + GlcNAc + a2-3Sia-Gc + a1-3Fuc", {"F":1, "H":2, "N":1, "G":1}, {"unknown": "?"}, [1233], "ext"]


#2 Fucose
LeBY = ["Lewis BY antigen", {"F":2, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]#b3-Gal type1 w a4-Fuc, a2-Fuc at Gal

#S+Fucose
sLeAX = ["Sialyl-Lewis AX antigen", {"F":1, "H":1, "N":1, "S":1}, {"unknown enzyme": "?"}, [0.0], "ext, ter, type1"]

#1 Fucose
Htype = ["Htype1 2 LNAc", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ter, type1"]
LeAX = ["Lewis AX antigen", {"F":1, "H":1, "N":1}, {"unknown enzyme": "?"}, [0.0], "ext, type1"]#b3-Gal type1 w a4-Fuc

#1 Sialic acid
Sia_LNAc = ["Sialylated LNAc", {"H":1, "N":1, "S":1}, {}, [376, 580, 792, 825], "ter, type1"]
SiaG_LNAc = ["SialylatedGc LNAc", {"H":1, "N":1, "G":1}, {}, [406, 610, 822, 855], "ter, type1"]
KDN_LNAc = ["SialylatedKDN LNAc", {"H":1, "N":1, "KDN":1}, {}, [406, 610, 822, 855], "ter, type1"]


#zf_GalExtension = ["a1-4 Galx2 + GlcNAc", {"H":2, "N":1}, {"unknown": "?"}, [668], "ext"]
#Sia6_LacDiNAc = ["a2-6Sia_LacDiNAc", {"N":2}, {}, [505], "ext, ter"]


#general extension if there is more extending units available
LacNAc = ["LacNAc", {"H":1, "N":1}, {"B3GALT2":1, "B3GALT5":1}, [464], "ext, type1"]

#orphan units adds in the end
Neu5Ac = ["Neu5Ac sialic acid", {"S": 1}, {"CMAS": 1}, [376],  "ext, ter"]
Neu5Gc = ["Neu5Gc sialic acid", {"G": 1}, {"CMAH": 1}, [406], "ext, ter"]
#KDN = ["KDN sialic acid", {"KDN": 1}, {"CMP-KDN synthetase": 1}, [335], "ter"]
alphaGal = ["a1-4 Galx2 + GlcNAc orphan count", {"H":1}, {"unknown": "?"}, [668], "ter"]
#5Ac and 5Gc can be added on any arms with Sia at terminal

demo_epitopes = [
    Bisecting_NG_core, 
    Biante_NG_core,
    Triante_NG_core,
    Tetraante_NG_core,
    Hybrid_NG_core,
    corefuc,
    LeBY,
    sLeAX,
    sLeAX,
    Htype,
    LeAX,
    Sia_LNAc,
    SiaG_LNAc,
    KDN_LNAc,
    LacNAc,
    Neu5Ac,
    Neu5Gc,
    alphaGal,
]
print(demo_epitopes)

this cells includes demo epitopes that don't check enzyme and peaklist
[['Bisecting NG core', {'H': 3, 'N': 3}, {'Bisecting enzyme': 1}, [], 'core'], ['Bi-antennary N-glycan core', {'H': 5, 'N': 4}, {}, [], 'core'], ['Tri-antennary N-glycan core', {'H': 6, 'N': 5}, {}, [], 'core'], ['Tetra-antennary N-glycan core', {'H': 7, 'N': 6}, {}, [], 'core'], ['Bi-antennary hybrid N-glycan core', {'H': 6, 'N': 3}, {}, [], 'core'], ['N-glycan core Fucosylation', {'F': 1}, {'FUT8': 1}, [452.249, 697.3753], 'ter, coreter'], ['Lewis BY antigen', {'F': 2, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ter, type1'], ['Sialyl-Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1, 'S': 1}, {'unknown enzyme': '?'}, [0.0], 'ext, ter, type1'], ['Sialyl-Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1, 'S': 1}, {'unknown enzyme': '?'}, [0.0], 'ext, ter, type1'], ['Htype1 2 LNAc', {'F': 1, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ter, type1'], ['Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}

In [44]:
import re

#finding and defining epitopes into groups

print("testing cores")
coreelements = []
for i in demo_epitopes:
    if ("core" in i):
        coreelements.append(i)
        print(i)
        
extelements = []
print("testing extentives")
for i in demo_epitopes:
    if re.search("ext", i[4]):
        extelements.append(i)
        print(i)
        
        
terelements = []
print("testing terminal elements")
for i in demo_epitopes:
    if re.search("ter", i[4]):
        terelements.append(i)
        print(i)

#print(f"Core elements are {coreelements}")
#print(f"Extensive elements are {extelements}")
#print(f"Terminal elements are {terelements}")

testing cores
['Bisecting NG core', {'H': 3, 'N': 3}, {'Bisecting enzyme': 1}, [], 'core']
['Bi-antennary N-glycan core', {'H': 5, 'N': 4}, {}, [], 'core']
['Tri-antennary N-glycan core', {'H': 6, 'N': 5}, {}, [], 'core']
['Tetra-antennary N-glycan core', {'H': 7, 'N': 6}, {}, [], 'core']
['Bi-antennary hybrid N-glycan core', {'H': 6, 'N': 3}, {}, [], 'core']
testing extentives
['Sialyl-Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1, 'S': 1}, {'unknown enzyme': '?'}, [0.0], 'ext, ter, type1']
['Sialyl-Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1, 'S': 1}, {'unknown enzyme': '?'}, [0.0], 'ext, ter, type1']
['Lewis AX antigen', {'F': 1, 'H': 1, 'N': 1}, {'unknown enzyme': '?'}, [0.0], 'ext, type1']
['LacNAc', {'H': 1, 'N': 1}, {'B3GALT2': 1, 'B3GALT5': 1}, [464], 'ext, type1']
['Neu5Ac sialic acid', {'S': 1}, {'CMAS': 1}, [376], 'ext, ter']
['Neu5Gc sialic acid', {'G': 1}, {'CMAH': 1}, [406], 'ext, ter']
testing terminal elements
['N-glycan core Fucosylation', {'F': 1}, {'FUT8': 1}, [452.249, 

In [23]:
import ast

def predcomplsttodict(complist):
    predcomp = {}
    if len(complist) >= 5:
        predcomp["F"] = complist[0]
        predcomp["H"] = complist[1]
        predcomp["N"] = complist[2]
        predcomp["S"] = complist[3]
        predcomp["G"] = complist[4]
        if len(complist) == 6:
            predcomp["KDN"] = complist[5]
    print(f"predicted composition dict is {predcomp}")
    return(predcomp)

def decuder(specific_epitopes, spectrapredcomp):
    print("...")
    print("Try to assign NG core decreasing")
    calc_comp = spectrapredcomp #store to another dict to avoid updating on original data needed later
    
    group_NGcore = 
    #cdict = {key: calc[key] - bdict.get(key, 0) for key in adict}
    

def deducecompositiona(specific_epitopes, speccompositions):
    print("developing")
    #create index first
    index = 0
    for i in range(len(speccompositions)):
        #print(speccompositions.loc[i])
        #print(speccompositions.loc[i]["predictedcomp"])
        strlist1 = speccompositions.loc[i]["predictedcomp"]
        #list1 = strlist1.split(',')
        list2 = ast.literal_eval(strlist1)
        #print(type(list1))
        #print(list1)
        #print(type(list2))
        print(list2)
        print(len(list2))
        for j in range(len(list2)):
            #print(type(list2[j][1]))
            #print(list2[j][1])
            complist = ast.literal_eval(list2[j][1]) #convert string to tuple
            spectrapredcomp = predcomplsttodict(complist) #convert tuple to dict (add KDN if structure has "6" components)
            #this extracted tuple value is what we're going to calculate...or manipulate? in real.
            #decuder(specific_epitopes, spectrapredcomp)
            decuder(demo_epitopes, spectrapredcomp)
    
#read 
sample1=pd.read_csv('Annotated_revised_zfNGintenstine_20230612_cloud.csv', sep='\t')

deduceinput = findpossibleepitope(c, zfdemo)
speccompositions = sample1.head(24)
                   
deducecompositiona(deduceinput, speccompositions)

#https://stackoverflow.com/questions/36459969/how-to-convert-a-list-to-a-dictionary-with-indexes-as-values isn't working

There is no enzyme for this epitope Bisecting NG core
There is no enzyme for this epitope First and elongation of Galactose at NG 
There is no enzyme for this epitope N-glycan core Fucosylation
There is no enzyme for this epitope Type 1 LacNAc
There is no enzyme for this epitope Type 2 LacNAc
developing
[(1639.854, '(0, 3, 4, 0, 0, 0)')]
1
<class 'str'>
(0, 3, 4, 0, 0, 0)
predicted composition dict is {'F': 0, 'H': 3, 'N': 4, 'S': 0, 'G': 0, 'KDN': 0}
...
[(1639.854, '(0, 3, 4, 0, 0, 0)')]
1
<class 'str'>
(0, 3, 4, 0, 0, 0)
predicted composition dict is {'F': 0, 'H': 3, 'N': 4, 'S': 0, 'G': 0, 'KDN': 0}
...
[(1639.854, '(0, 3, 4, 0, 0, 0)')]
1
<class 'str'>
(0, 3, 4, 0, 0, 0)
predicted composition dict is {'F': 0, 'H': 3, 'N': 4, 'S': 0, 'G': 0, 'KDN': 0}
...
[(1639.854, '(0, 3, 4, 0, 0, 0)')]
1
<class 'str'>
(0, 3, 4, 0, 0, 0)
predicted composition dict is {'F': 0, 'H': 3, 'N': 4, 'S': 0, 'G': 0, 'KDN': 0}
...
[(1639.854, '(0, 3, 4, 0, 0, 0)')]
1
<class 'str'>
(0, 3, 4, 0, 0, 0)
predi

In [8]:
#calculations only works on dict, not list or tuple... meaning we worked in vain on making them list
#https://stackoverflow.com/questions/17671875/how-to-subtract-values-from-dictionaries
adict = {"F":1, "H":5 , "N": 4, "S":1}
bdict = {"H":3, "N":2}
cdict = {key: adict[key] - bdict.get(key, 0) for key in adict}
print(cdict)

{'F': 1, 'H': 2, 'N': 2, 'S': 1}
